In [1]:
# =============================================================================
# CS-781 Phase 2 — L-NER, Experiment 2: Arch #5 (Prefix-LM hybrid) used as a
# BIDIRECTIONAL tagger, full fine-tuning, BIO head.
#
# Why Arch #5: it was pretrained with bidirectional attention inside its
# prefix, so feeding an NER window as the prefix uses attention the way it was
# pretrained. This lets a party name see the role word that FOLLOWS it
# ("... APPELLANT(S)"), which a causal model cannot do.
#
# Changes vs Experiment 1 (all else is kept the same so the comparison is clean):
#   * bidirectional backbone (Arch #5), windows of prefix_len tokens, stride 256
#   * overlapping windows merged by CENTRALITY (token closest to window middle wins)
#   * Viterbi decoding with BIO transition constraints (compared against argmax)
#   * error-analysis table (correct / boundary / label confusion / missed / spurious)
#   * HOLDOUT_DOCS = 0 trains on all 70 docs for the final submission
#
# Run in Kaggle: paste each "# %%" block as its own cell, in order.
# =============================================================================

In [2]:
# %% [1] Imports & HF login
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json, math, random, time, zipfile, re, itertools
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Device: cuda


In [3]:
# %% [2] Config
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)

ARTIFACT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-token5"          # tokenizer
# !!! EDIT: Arch #5 checkpoint attached as a Kaggle input (model_best.pt from the resumable run is best)
ARCH5_CKPT_PATH = "/kaggle/input/datasets/akashmalgav2/phase2-model5/model_best_5.pt"
# Used only if the checkpoint file has no "config" entry:
DEFAULT_CFG = dict(vocab_size=32000, n_layer=12, n_head=12, n_embd=768,
                   prefix_len=512, cont_len=160, ffn_mult=4)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_TEXT_FILES = ["Phase_2_LNER/Train/fold1-text.json", "Phase_2_LNER/Train/fold2-text.json"]
TRAIN_GOLD_FILES = ["Phase_2_LNER/Train/fold1-gold.json", "Phase_2_LNER/Train/fold2-gold.json"]
LABELS_FILE = "Phase_2_LNER/Train/labels.txt"
TEST_TEXT_FILE = "Phase_2_LNER/Test/fold3-text.json"

HOLDOUT_DOCS = 10             # set to 0 for the FINAL run on all 70 docs (use epoch count found with holdout)
STRIDE = 256
EPOCHS = 6
BATCH_SIZE = 16
EVAL_BATCH = 8
LR_BACKBONE = 5e-5
LR_HEAD = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
DROPOUT_FT = 0.1
O_WEIGHT = 1.0

OUT_DIR = "/kaggle/working/cs781_phase2_exp2"
os.makedirs(OUT_DIR, exist_ok=True)
BEST_PATH = os.path.join(OUT_DIR, "ner_best.pt")
PRED_PATH = os.path.join(OUT_DIR, "prediction.json")
ZIP_PATH = os.path.join(OUT_DIR, "submission_phase2_exp2.zip")

In [4]:
# %% [3] Load Phase-2 data
def hf_local(path):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=path, repo_type="dataset", token=HF_TOKEN)

LABELS = [x.strip() for x in open(hf_local(LABELS_FILE), encoding="utf-8") if x.strip()]
assert len(LABELS) == 12, LABELS

ID2TAG = {0: ("O", None)}
for i, lab in enumerate(LABELS):
    ID2TAG[1 + 2 * i] = ("B", lab)
    ID2TAG[2 + 2 * i] = ("I", lab)
NUM_TAGS = len(ID2TAG)
B_ID = {lab: 1 + 2 * i for i, lab in enumerate(LABELS)}
I_ID = {lab: 2 + 2 * i for i, lab in enumerate(LABELS)}

def norm_entity(e):
    if isinstance(e, (list, tuple)):
        s, en, lab = e[0], e[1], e[2]
        txt = e[3] if len(e) > 3 else None
    elif isinstance(e, dict):
        def pick(*ks):
            for k in ks:
                if k in e:
                    return e[k]
            return None
        s = pick("start", "start_char", "start_offset", "begin")
        en = pick("end", "end_char", "end_offset")
        lab = pick("label", "type", "entity", "tag")
        txt = pick("text", "entity_text", "span")
    else:
        raise ValueError(f"Unrecognised entity format: {e!r}")
    return int(s), int(en), str(lab), txt

texts, gold = {}, {}
for tf, gf in zip(TRAIN_TEXT_FILES, TRAIN_GOLD_FILES):
    t = json.load(open(hf_local(tf), encoding="utf-8"))
    g = json.load(open(hf_local(gf), encoding="utf-8"))
    for k, v in t.items():
        assert k not in texts, f"duplicate doc id {k}"
        texts[k] = v
    for k, v in g.items():
        if isinstance(v, dict):
            v = v.get("entities") or v.get("annotations") or v.get("labels") or []
        gold[k] = [norm_entity(e) for e in v]
for k in texts:
    gold.setdefault(k, [])
lab_counts = Counter(l for ents in gold.values() for (_, _, l, _) in ents)
print("Docs:", len(texts), "| entities per label:", dict(lab_counts))

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

Docs: 70 | entities per label: {'RESP': 381, 'DATE': 1575, 'COURT': 1013, 'WIT': 187, 'APP': 407, 'STAT': 1260, 'A.COUNSEL': 195, 'PREC': 1183, 'JUDGE': 245, 'R.COUNSEL': 184, 'CASENO': 760, 'AUTH': 1003}


In [5]:
# %% [4] Tokenizer (same BPE as Phase 1) + tokenization WITH exact char offsets
with open(f"{ARTIFACT_PATH}/legal_bpe.json", "r", encoding="utf-8") as f:
    _tok = json.load(f)
vocab = _tok["itos"]
merges = [tuple(p) for p in _tok["merges"]]
token_to_id = {tok: i for i, tok in enumerate(vocab)}
merges_rank = {p: i for i, p in enumerate(merges)}
PAD_ID, UNK_ID = token_to_id["<pad>"], token_to_id["<unk>"]
BOS_ID, EOS_ID = token_to_id["<s>"], token_to_id["</s>"]
VOCAB_SIZE = len(vocab)
EOW = "\u0100"
WORD_RE = re.compile(r"\S+")

def word_to_symbols(word):
    return tuple(chr(b) for b in word.encode("utf-8", "ignore")) + (EOW,)

def bpe_encode_word(word):
    symbols = list(word_to_symbols(word))
    while len(symbols) > 1:
        pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
        ranked = [(merges_rank[p], p) for p in pairs if p in merges_rank]
        if not ranked:
            break
        _, best = min(ranked)
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == best[0] and symbols[i + 1] == best[1]:
                new_symbols.append(best[0] + best[1]); i += 2
            else:
                new_symbols.append(symbols[i]); i += 1
        symbols = new_symbols
    return symbols

def tokenize_with_offsets(text):
    ids, offs = [], []
    for m in WORD_RE.finditer(text):
        word, w0 = m.group(), m.start()
        b2c = []
        for ci, ch in enumerate(word):
            for _ in ch.encode("utf-8", "ignore"):
                b2c.append(ci)
        bpos = 0
        for sym in bpe_encode_word(word):
            nb = sum(1 for ch in sym if ch != EOW)
            if nb == 0 or not b2c:
                ce = w0 + len(word)
                offs.append((ce, ce))
            else:
                sb = min(bpos, len(b2c) - 1)
                eb = min(bpos + nb, len(b2c))
                offs.append((w0 + b2c[sb], w0 + b2c[eb - 1] + 1))
            bpos += nb
            ids.append(token_to_id.get(sym, UNK_ID))
    return ids, offs

In [6]:
# %% [5] Per-document records: token ids, offsets, BIO labels
def encode_doc(text, spans=None):
    ids, offs = tokenize_with_offsets(text)
    nz = [i for i, (a, b) in enumerate(offs) if b > a]
    rec = {"ids": ids, "offs": offs, "nz": nz, "len": len(text)}
    if spans is None:
        return rec
    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    char_sid = [-1] * len(text)
    for k, (s, e, lab, _) in enumerate(spans):
        for c in range(max(0, s), min(len(text), e)):
            char_sid[c] = k
    labels = [-100] * len(ids)
    span_rng = {}
    prev_sid = -1
    for t in nz:
        cs, ce = offs[t]
        sid = -1
        for c in range(cs, ce):
            if char_sid[c] >= 0:
                sid = char_sid[c]
                break
        if sid < 0:
            labels[t] = 0
        else:
            lab = spans[sid][2]
            labels[t] = B_ID[lab] if sid != prev_sid else I_ID[lab]
            lo, hi = span_rng.get(sid, (cs, ce))
            span_rng[sid] = (min(lo, cs), max(hi, ce))
        prev_sid = sid
    rec.update(labels=labels, spans=spans, span_rng=span_rng)
    return rec

recs = {k: encode_doc(texts[k], gold[k]) for k in texts}
print(f"Encoded {len(recs)} docs | tokens: {sum(len(r['ids']) for r in recs.values()):,}")

Encoded 70 docs | tokens: 421,043


In [7]:
# %% [6] Oracle check -> picks the punctuation trim rule (same procedure as Experiment 1)
def trim_span(text, s, e, lead, trail):
    while s < e and (text[s].isspace() or text[s] in lead):
        s += 1
    while e > s and (text[e - 1].isspace() or text[e - 1] in trail):
        e -= 1
    return s, e

LEAD_OPTS = ["", "(\"'[\u201c\u2018"]
TRAIL_OPTS = ["", ",;", ",;:", ",;:.", ",;:.)\"'\u201d\u2019]", ",;:)\"'\u201d\u2019]"]
total_spans = sum(len(r["spans"]) for r in recs.values())
results = []
for lead, trail in itertools.product(LEAD_OPTS, TRAIL_OPTS):
    ok = 0
    for k, r in recs.items():
        for sid, (cs, ce) in r["span_rng"].items():
            s, e = r["spans"][sid][0], r["spans"][sid][1]
            ok += int(trim_span(texts[k], cs, ce, lead, trail) == (s, e))
    results.append((ok / max(1, total_spans), lead, trail))
results.sort(key=lambda x: -x[0])
ORACLE_RATE, TRIM_LEAD, TRIM_TRAIL = results[0]
print(f"Oracle exact-match ceiling {ORACLE_RATE:.4f} | TRIM_LEAD={TRIM_LEAD!r} TRIM_TRAIL={TRIM_TRAIL!r}")


Oracle exact-match ceiling 0.9402 | TRIM_LEAD='("\'[“‘' TRIM_TRAIL=',;:.'


In [8]:
# %% [7] Model: Arch #5 backbone (bidirectional inside the prefix) + token-classification head
def build_prefix_lm_mask(total_len, prefix_len, device):
    idx = torch.arange(total_len, device=device)
    is_prefix_col = idx < prefix_len
    rows, cols = idx.unsqueeze(1), idx.unsqueeze(0)
    allowed = is_prefix_col.unsqueeze(0).expand(total_len, total_len).clone()
    cont_causal = (rows >= cols) & (cols >= prefix_len) & (rows >= prefix_len)
    return allowed | cont_causal

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
    def forward(self, x, attn_mask):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

class SegmentGatedExpertMLP(nn.Module):
    def __init__(self, n_embd, mult, dropout):
        super().__init__()
        hidden = mult * n_embd
        self.expert_prefix_leaning = nn.Sequential(nn.Linear(n_embd, hidden), nn.GELU(), nn.Linear(hidden, n_embd))
        self.expert_continuation_leaning = nn.Sequential(nn.Linear(n_embd, hidden), nn.GELU(), nn.Linear(hidden, n_embd))
        self.gate_proj = nn.Linear(n_embd, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_proj(x))
        return self.drop(gate * self.expert_prefix_leaning(x) + (1 - gate) * self.expert_continuation_leaning(x))

class Block(nn.Module):
    def __init__(self, n_embd, n_head, ffn_mult, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = SegmentGatedExpertMLP(n_embd, ffn_mult, dropout)
    def forward(self, x, attn_mask):
        x = x + self.attn(self.ln1(x), attn_mask)
        return x + self.mlp(self.ln2(x))

class PrefixLMHybrid(nn.Module):
    def __init__(self, vocab_size, n_layer, n_head, n_embd, prefix_len, cont_len, ffn_mult, dropout, pad_id):
        super().__init__()
        self.prefix_len, self.cont_len = prefix_len, cont_len
        self.total_len = prefix_len + cont_len
        self.pad_id = pad_id
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(self.total_len, n_embd)
        self.segment_emb = nn.Embedding(2, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, ffn_mult, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.tok_emb.weight = self.head.weight
        self.register_buffer("struct_mask", build_prefix_lm_mask(self.total_len, prefix_len, "cpu"), persistent=False)

    def forward_hidden(self, seq_ids, key_valid):
        B, T = seq_ids.shape
        assert T <= self.prefix_len, "window must fit inside the bidirectional prefix"
        pos = torch.arange(T, device=seq_ids.device).unsqueeze(0)
        seg = (torch.arange(T, device=seq_ids.device) >= self.prefix_len).long().unsqueeze(0)
        x = self.drop(self.tok_emb(seq_ids) + self.pos_emb(pos) + self.segment_emb(seg))
        struct = self.struct_mask.to(seq_ids.device)[:T, :T]          # all True for T <= prefix_len
        attn_mask = struct.unsqueeze(0).unsqueeze(0) & key_valid[:, None, None, :]
        for blk in self.blocks:
            x = blk(x, attn_mask)
        return self.ln_f(x)

class NERModel(nn.Module):
    def __init__(self, backbone, n_embd, num_tags, dropout, pad_id):
        super().__init__()
        self.backbone, self.pad_id = backbone, pad_id
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(n_embd, num_tags)
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    def forward(self, idx):
        key_valid = idx != self.pad_id
        return self.classifier(self.drop(self.backbone.forward_hidden(idx, key_valid)))

ckpt = torch.load(ARCH5_CKPT_PATH, map_location="cpu")
cfg = ckpt.get("config", DEFAULT_CFG)
state = ckpt.get("model_state_dict", ckpt.get("model"))
assert state is not None, f"No model weights found in checkpoint; keys: {list(ckpt.keys())}"
assert cfg["vocab_size"] == VOCAB_SIZE, (cfg["vocab_size"], VOCAB_SIZE)
backbone = PrefixLMHybrid(cfg["vocab_size"], cfg["n_layer"], cfg["n_head"], cfg["n_embd"],
                          cfg["prefix_len"], cfg["cont_len"], cfg["ffn_mult"], DROPOUT_FT, PAD_ID)
backbone.load_state_dict(state)
model = NERModel(backbone, cfg["n_embd"], NUM_TAGS, DROPOUT_FT, PAD_ID).to(device)
CONTENT = cfg["prefix_len"] - 1            # 1 slot for <s>
print(f"Loaded Arch #5 backbone (step {ckpt.get('step')}, val_loss {ckpt.get('val_loss')}) | "
      f"NER params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M | window = {CONTENT+1} tokens")


Loaded Arch #5 backbone (step 3250, val_loss 2.723325307641062) | NER params: 166.8M | window = 512 tokens


In [9]:
# %% [8] Windows, datasets, split
def make_windows(n, content, stride):
    if n <= content:
        return [(0, n)]
    starts = list(range(0, n - content + 1, stride))
    if starts[-1] + content < n:
        starts.append(n - content)
    return [(s, s + content) for s in starts]

doc_ids = sorted(recs.keys())
random.Random(SEED).shuffle(doc_ids)
val_ids = doc_ids[:HOLDOUT_DOCS]
train_ids = doc_ids[HOLDOUT_DOCS:]
print(f"Train docs: {len(train_ids)} | held-out docs: {len(val_ids)}")

class WindowDataset(Dataset):
    def __init__(self, doc_keys):
        self.items = []
        for k in doc_keys:
            r = recs[k]
            for (s, e) in make_windows(len(r["ids"]), CONTENT, STRIDE):
                x = [BOS_ID] + r["ids"][s:e]
                y = [-100] + r["labels"][s:e]
                pad = CONTENT + 1 - len(x)
                self.items.append((x + [PAD_ID] * pad, y + [-100] * pad))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        x, y = self.items[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

train_ds = WindowDataset(train_ids)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
print("Training windows:", len(train_ds), "| steps/epoch:", len(train_loader))

Train docs: 60 | held-out docs: 10
Training windows: 1353 | steps/epoch: 85


In [10]:
# %% [9] Inference (centrality merge), decoding (argmax vs Viterbi), strict macro-F1
@torch.no_grad()
def predict_logp(net, rec):
    """Per-token log-probs over tags. Overlapping windows are merged by
    centrality: each token keeps the prediction from the window in which it is
    closest to the middle (most context on BOTH sides)."""
    net.eval()
    n = len(rec["ids"])
    wins = make_windows(n, CONTENT, STRIDE)
    store = torch.zeros(n, NUM_TAGS)
    best_c = torch.full((n,), -1, dtype=torch.long)
    for i in range(0, len(wins), EVAL_BATCH):
        chunk = wins[i:i + EVAL_BATCH]
        batch = torch.full((len(chunk), CONTENT + 1), PAD_ID, dtype=torch.long)
        for r_i, (s, e) in enumerate(chunk):
            batch[r_i, 0] = BOS_ID
            batch[r_i, 1:1 + e - s] = torch.tensor(rec["ids"][s:e], dtype=torch.long)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = net(batch.to(device))
        logp = F.log_softmax(logits.float(), dim=-1).cpu()
        for r_i, (s, e) in enumerate(chunk):
            L = e - s
            pos = torch.arange(L)
            cen = torch.minimum(pos, L - 1 - pos)
            idx = s + pos
            upd = cen > best_c[idx]
            store[idx[upd]] = logp[r_i, 1:1 + L][upd]
            best_c[idx[upd]] = cen[upd]
    return store

def build_transitions():
    NEG = -1e9
    trans = np.zeros((NUM_TAGS, NUM_TAGS), dtype=np.float32)
    for a in range(NUM_TAGS):
        for b in range(NUM_TAGS):
            tag_a, ent_a = ID2TAG[a]
            tag_b, ent_b = ID2TAG[b]
            if tag_b == "I" and not (tag_a in ("B", "I") and ent_a == ent_b):
                trans[a, b] = NEG                    # I-X is only valid right after B-X or I-X
    start = np.array([NEG if ID2TAG[b][0] == "I" else 0.0 for b in range(NUM_TAGS)], dtype=np.float32)
    return trans, start
TRANS, START = build_transitions()

def viterbi(logp):
    n = logp.shape[0]
    score = START + logp[0]
    back = np.zeros((n, NUM_TAGS), dtype=np.int64)
    for t in range(1, n):
        cand = score[:, None] + TRANS
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(n, dtype=np.int64)
    path[-1] = score.argmax()
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path

def tags_from_logp(logp, rec, mode):
    tags = np.zeros(logp.shape[0], dtype=np.int64)
    nz = rec["nz"]
    if not nz:
        return tags
    lp = logp[nz].numpy()
    tags[nz] = lp.argmax(1) if mode == "argmax" else viterbi(lp)
    return tags

def decode_spans(text, tags, rec):
    spans, cur = [], None
    for t in rec["nz"]:
        tid = int(tags[t])
        cs, ce = rec["offs"][t]
        if tid == 0:
            if cur:
                spans.append(cur); cur = None
            continue
        tag, ent = ID2TAG[tid]
        if cur is not None and tag == "I" and cur[2] == ent:
            cur[1] = ce
        else:
            if cur:
                spans.append(cur)
            cur = [cs, ce, ent]
    if cur:
        spans.append(cur)
    out = set()
    for s, e, ent in spans:
        s2, e2 = trim_span(text, s, e, TRIM_LEAD, TRIM_TRAIL)
        if e2 > s2:
            out.add((s2, e2, ent))
    return sorted(out)

def strict_macro(truth, pred):
    stats = {l: [0, 0, 0] for l in LABELS}
    for d in truth:
        g, p = set(truth[d]), set(pred.get(d, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}
            ps = {x for x in p if x[2] == l}
            stats[l][0] += len(gs & ps); stats[l][1] += len(ps - gs); stats[l][2] += len(gs - ps)
    P, R, F_, per = [], [], [], {}
    for l in LABELS:
        tp, fp, fn = stats[l]
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        P.append(pr); R.append(rc); F_.append(f); per[l] = (pr, rc, f)
    return sum(F_) / len(F_), sum(P) / len(P), sum(R) / len(R), per

def evaluate_heldout(net, keys):
    truth = {k: sorted({(s, e, l) for (s, e, l, _) in gold[k]}) for k in keys}
    preds = {"argmax": {}, "viterbi": {}}
    for k in keys:
        logp = predict_logp(net, recs[k])
        for mode in preds:
            preds[mode][k] = decode_spans(texts[k], tags_from_logp(logp, recs[k], mode), recs[k])
    return {m: strict_macro(truth, preds[m]) for m in preds}, truth, preds

def error_analysis(truth, pred):
    """Classify every gold span and every unmatched predicted span."""
    cats = {l: Counter() for l in LABELS}
    spurious = Counter()
    for d in truth:
        g, p = truth[d], pred[d]
        pset, gset = set(p), set(g)
        for (s, e, l) in g:
            if (s, e, l) in pset:
                cats[l]["correct"] += 1
            elif any(pl == l and ps < e and pe > s for ps, pe, pl in p):
                cats[l]["boundary"] += 1
            elif any(pl != l and ps < e and pe > s for ps, pe, pl in p):
                cats[l]["label_confusion"] += 1
            else:
                cats[l]["missed"] += 1
        for (ps, pe, pl) in p:
            if (ps, pe, pl) not in gset and not any(gs < pe and ge > ps for gs, ge, gl in g):
                spurious[pl] += 1
    print(f"{'label':10s} {'gold':>5s} {'correct':>8s} {'boundary':>9s} {'confused':>9s} {'missed':>7s} {'spurious':>9s}")
    for l in LABELS:
        c = cats[l]
        n = sum(c.values())
        print(f"{l:10s} {n:5d} {c['correct']:8d} {c['boundary']:9d} {c['label_confusion']:9d} {c['missed']:7d} {spurious[l]:9d}")


In [11]:

# %% [10] Training loop (full fine-tuning)
class_w = torch.ones(NUM_TAGS)
class_w[0] = O_WEIGHT
class_w = class_w.to(device)

head_params = list(model.classifier.parameters())
head_ids = {id(p) for p in head_params}
body_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.AdamW(
    [{"params": body_params, "lr": LR_BACKBONE, "base": LR_BACKBONE},
     {"params": head_params, "lr": LR_HEAD, "base": LR_HEAD}],
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.98))
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

total_steps = EPOCHS * len(train_loader)
warm = max(1, int(WARMUP_FRAC * total_steps))
def lr_factor(step):
    if step < warm:
        return step / warm
    prog = (step - warm) / max(1, total_steps - warm)
    return max(0.02, 0.5 * (1 + math.cos(math.pi * prog)))

history, best_f1, step, t0 = [], -1.0, 0, time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, n_b = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        for g in optimizer.param_groups:
            g["lr"] = g["base"] * lr_factor(step)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            logits = model(x)
        loss = F.cross_entropy(logits.float().view(-1, NUM_TAGS), y.view(-1), weight=class_w, ignore_index=-100)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item(); n_b += 1; step += 1
        if step % 50 == 0:
            print(f"  step {step}/{total_steps} | loss {run_loss/n_b:.4f} | {(time.time()-t0)/60:.1f}m")
    rec_h = {"epoch": epoch, "train_loss": run_loss / max(1, n_b)}
    if val_ids:
        out, _, _ = evaluate_heldout(model, val_ids)
        rec_h.update(f1_argmax=out["argmax"][0], f1_viterbi=out["viterbi"][0])
        print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} | held-out strict macro-F1: "
              f"argmax {out['argmax'][0]:.4f} | viterbi {out['viterbi'][0]:.4f} "
              f"(P {out['viterbi'][1]:.4f} R {out['viterbi'][2]:.4f}) | {(time.time()-t0)/60:.1f}m")
        score = out["viterbi"][0]
        if score > best_f1:
            best_f1 = score
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": score}, BEST_PATH)
            print("  -> new best checkpoint saved")
    else:
        print(f"[epoch {epoch}] loss {rec_h['train_loss']:.4f} (no held-out; final run on all docs)")
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_f1": None}, BEST_PATH)   # keep last epoch
    history.append(rec_h)

json.dump(history, open(os.path.join(OUT_DIR, "history.json"), "w"), indent=2)


  step 50/510 | loss 0.7946 | 0.5m
[epoch 1] loss 0.5781 | held-out strict macro-F1: argmax 0.0348 | viterbi 0.0560 (P 0.0732 R 0.0499) | 0.9m
  -> new best checkpoint saved
  step 100/510 | loss 0.2488 | 1.1m
  step 150/510 | loss 0.2128 | 1.7m
[epoch 2] loss 0.2078 | held-out strict macro-F1: argmax 0.0633 | viterbi 0.0837 (P 0.0957 R 0.0798) | 2.0m
  -> new best checkpoint saved
  step 200/510 | loss 0.1723 | 2.4m
  step 250/510 | loss 0.1614 | 2.9m
[epoch 3] loss 0.1608 | held-out strict macro-F1: argmax 0.0648 | viterbi 0.0887 (P 0.0966 R 0.0854) | 3.0m
  -> new best checkpoint saved
  step 300/510 | loss 0.1373 | 3.6m
[epoch 4] loss 0.1336 | held-out strict macro-F1: argmax 0.0752 | viterbi 0.1072 (P 0.1135 R 0.1057) | 4.1m
  -> new best checkpoint saved
  step 350/510 | loss 0.1217 | 4.2m
  step 400/510 | loss 0.1176 | 4.8m
[epoch 5] loss 0.1170 | held-out strict macro-F1: argmax 0.0758 | viterbi 0.1084 (P 0.1145 R 0.1093) | 5.2m
  -> new best checkpoint saved
  step 450/510 | l

In [12]:
# %% [11] Per-label scores + error analysis (only meaningful when HOLDOUT_DOCS > 0)
best = torch.load(BEST_PATH, map_location="cpu")
model.load_state_dict(best["model"])
DECODE_MODE = "viterbi"
if val_ids:
    out, val_truth, val_preds = evaluate_heldout(model, val_ids)
    DECODE_MODE = max(out, key=lambda m: out[m][0])
    print(f"Best epoch {best['epoch']} | argmax F1 {out['argmax'][0]:.4f} | viterbi F1 {out['viterbi'][0]:.4f} -> using {DECODE_MODE}")
    per = out[DECODE_MODE][3]
    for l in LABELS:
        print(f"  {l:10s} P={per[l][0]:.3f} R={per[l][1]:.3f} F1={per[l][2]:.3f}  (gold n={lab_counts.get(l,0)})")
    print()
    error_analysis(val_truth, val_preds[DECODE_MODE])
    json.dump({k: [list(x) for x in v] for k, v in val_truth.items()}, open(os.path.join(OUT_DIR, "val_truth.json"), "w"))
    json.dump({k: [list(x) for x in v] for k, v in val_preds[DECODE_MODE].items()}, open(os.path.join(OUT_DIR, "val_pred.json"), "w"))
else:
    print("No held-out docs: decoding with Viterbi.")

Best epoch 6 | argmax F1 0.0759 | viterbi F1 0.1163 -> using viterbi
  APP        P=0.000 R=0.000 F1=0.000  (gold n=407)
  RESP       P=0.000 R=0.000 F1=0.000  (gold n=381)
  A.COUNSEL  P=0.059 R=0.038 F1=0.046  (gold n=195)
  R.COUNSEL  P=0.028 R=0.038 F1=0.032  (gold n=184)
  JUDGE      P=0.000 R=0.000 F1=0.000  (gold n=245)
  WIT        P=0.000 R=0.000 F1=0.000  (gold n=187)
  AUTH       P=0.263 R=0.110 F1=0.155  (gold n=1003)
  COURT      P=0.600 R=0.709 F1=0.650  (gold n=1013)
  STAT       P=0.204 R=0.136 F1=0.163  (gold n=1260)
  PREC       P=0.050 R=0.071 F1=0.059  (gold n=1183)
  DATE       P=0.278 R=0.275 F1=0.276  (gold n=1575)
  CASENO     P=0.016 R=0.012 F1=0.014  (gold n=760)

label       gold  correct  boundary  confused  missed  spurious
APP           78        0         2        19      57        20
RESP          58        0         9         9      40        10
A.COUNSEL     53        2         2        19      30        17
R.COUNSEL     53        2         8         9

In [13]:
# %% [12] Predict the test set (fold3) -> prediction.json -> zip
test_texts = json.load(open(hf_local(TEST_TEXT_FILE), encoding="utf-8"))
print("Test docs:", len(test_texts), "| decode mode:", DECODE_MODE)
test_pred = {}
t0 = time.time()
for i, (k, text) in enumerate(test_texts.items()):
    rec = encode_doc(text)
    spans = decode_spans(text, tags_from_logp(predict_logp(model, rec), rec, DECODE_MODE), rec)
    test_pred[k] = [[s, e, l] for (s, e, l) in spans]
    if (i + 1) % 10 == 0:
        print(f"  predicted {i+1}/{len(test_texts)} | {(time.time()-t0)/60:.1f}m")

assert set(test_pred) == set(test_texts), "ID mismatch"
for k, spans in test_pred.items():
    n = len(test_texts[k])
    for s, e, l in spans:
        assert 0 <= s < e <= n and l in LABELS, (k, s, e, l)
print("Total predicted entities:", sum(len(v) for v in test_pred.values()),
      "| per doc:", round(sum(len(v) for v in test_pred.values()) / len(test_pred), 1),
      "(training gold averages ~120 per doc)")
ex_k = next(k for k, v in test_pred.items() if v)
print("Sample:", [(s, e, l, test_texts[ex_k][s:e]) for s, e, l in test_pred[ex_k][:5]])

with open(PRED_PATH, "w", encoding="utf-8") as f:
    json.dump(test_pred, f)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(PRED_PATH, arcname="prediction.json")
print("Codabench submission ready:", ZIP_PATH)

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Test docs: 35 | decode mode: viterbi
  predicted 10/35 | 0.1m
  predicted 20/35 | 0.1m
  predicted 30/35 | 0.2m
Total predicted entities: 3890 | per doc: 111.1 (training gold averages ~120 per doc)
Sample: [(70, 98, 'CASENO', 'CIVIL APPEAL NO.6580 OF 2008'), (149, 151, 'DATE', '15'), (264, 277, 'CASENO', 'APPEAL NO.658'), (337, 339, 'DATE', '15'), (517, 531, 'CASENO', 'NO. 30 OF 2007')]
Codabench submission ready: /kaggle/working/cs781_phase2_exp2/submission_phase2_exp2.zip
